In [1]:
import nb_setup
import os
import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from pathlib import Path
import json
from sklearn.metrics import f1_score

from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from transformers import get_linear_schedule_with_warmup
from collections import defaultdict

from src.data import load_tokenised_data
from src.answerability.evaluation import evaluate_answerability_predictions
from src.integrated_qa import (
    build_qa_features,
    QATrainDataset,
    train_collate_fn,
    val_collate_fn,
    f1_score_squad,
    exact_match_score,
)
from src.utils import enforce_reproducibility
from evaluate import load as load_metric
from src.translator import NLLBTranslator, LANGUAGE_MAP

In [2]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

enforce_reproducibility()

df_train_tokenised, df_val_tokenised = load_tokenised_data()


In [3]:
def translate_dataframe(df, translator, batch_size=32):
    df = df.copy()
    df["original_question"] = df["question"]
    df["question"] = ""

    for lang in LANGUAGE_MAP:
        indices = df.index[df["lang"] == lang]
        questions = df.loc[indices, "original_question"].tolist()

        if not questions:
            continue

        translations = translator.translate_batch(
            questions,
            source_lang=lang,
            batch_size=batch_size,
        )

        df.loc[indices, "question"] = translations

        print(f"Translated {len(indices)} questions from {lang}")

    return df

In [4]:
translator = NLLBTranslator()
df_train_translated = translate_dataframe(df_train_tokenised, translator)
df_val_translated = translate_dataframe(df_val_tokenised, translator)

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

Translating ar:   0%|          | 0/80 [00:00<?, ?it/s]

Translated 2558 questions from ar


Translating ko:   0%|          | 0/76 [00:00<?, ?it/s]

Translated 2422 questions from ko


...

Translated 415 questions from ar


Translating ko:   0%|          | 0/12 [00:00<?, ?it/s]

Translated 356 questions from ko


Translating te:   0%|          | 0/12 [00:00<?, ?it/s]

Translated 384 questions from te


In [5]:
squad_metric = load_metric("squad_v2")


def compute_squad(*, references, predictions):
    return squad_metric.compute(references=references, predictions=predictions)


In [6]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 384
DOC_STRIDE = 128

SAVE_PATH = "../checkpoints/integrated_qa_translated"
RUN_TRAINING = False  # False: load a saved fine-tuned model and only evaluate

if RUN_TRAINING:
    model = AutoModelForQuestionAnswering.from_pretrained(
        MODEL_NAME,
        attn_implementation="eager",
    )
else:
    model = AutoModelForQuestionAnswering.from_pretrained(
        SAVE_PATH,
        attn_implementation="eager",
    )

if RUN_TRAINING:
    for param in model.roberta.parameters():
        param.requires_grad = False

    for layer in model.roberta.encoder.layer[-4:]:
        for param in layer.parameters():
            param.requires_grad = True

    for param in model.qa_outputs.parameters():
        param.requires_grad = True

model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME if RUN_TRAINING else SAVE_PATH,
    use_fast=True,
)

if RUN_TRAINING:
    train_features = build_qa_features(
        df_train_translated,
        tokenizer,
        max_length=MAX_LENGTH,
        doc_stride=DOC_STRIDE,
    )

validation_features = build_qa_features(
    df_val_translated,
    tokenizer,
    max_length=MAX_LENGTH,
    doc_stride=DOC_STRIDE,
)

if RUN_TRAINING:
    train_dataset = QATrainDataset(train_features)

    train_dl = DataLoader(
        train_dataset,
        collate_fn=train_collate_fn,
        shuffle=True,
        batch_size=4,
    )


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
lr = 2e-5
n_epochs = 20
weight_decay = 0.01
warmup_steps = 200

if RUN_TRAINING:
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [
                p
                for n, p in model.named_parameters()
                if p.requires_grad and not any(nd in n for nd in no_decay)
            ],
            "weight_decay": weight_decay,
        },
        {
            "params": [
                p
                for n, p in model.named_parameters()
                if p.requires_grad and any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]

    optimizer = AdamW(optimizer_grouped_parameters, lr=lr)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=n_epochs * len(train_dl),
    )

    total_steps = n_epochs * len(train_dl)

    print("Steps per epoch:", len(train_dl))
    print("Total steps:", total_steps)
    print("Warmup steps:", warmup_steps)


In [8]:
def qa_loss_collate_fn(batch):
    keys = ["input_ids", "attention_mask", "start_positions", "end_positions"]
    return {
        key: torch.tensor([item[key] for item in batch], dtype=torch.long)
        for key in keys
    }


def evaluate_validation_loss(model, val_loss_dl, device):
    model.eval()
    total_loss = 0.0
    total_windows = 0

    with torch.inference_mode():
        for batch in tqdm(val_loss_dl, desc="Validation loss", leave=False):
            batch = {key: value.to(device) for key, value in batch.items()}
            outputs = model(**batch)
            n = batch["input_ids"].shape[0]
            total_loss += outputs.loss.item() * n
            total_windows += n

    return total_loss / total_windows


def train(
    model,
    train_dl,
    val_loss_dl,
    optimizer,
    scheduler,
    n_epochs,
    device,
    patience=3,
    checkpoint_path="../checkpoints/integrated_qa_translated/best_state_dict.pt",
):
    losses = []
    history = []
    best_val_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

    for epoch in range(n_epochs):
        model.train()
        loss_epoch = []

        for batch in tqdm(train_dl, desc=f"Epoch {epoch + 1}/{n_epochs}"):
            optimizer.zero_grad()
            batch = {key: value.to(device) for key, value in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            losses.append(loss.item())
            loss_epoch.append(loss.item())

            loss.backward()
            optimizer.step()
            scheduler.step()

        train_loss = float(np.mean(loss_epoch))
        val_loss = evaluate_validation_loss(model, val_loss_dl, device)
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_loss": val_loss,
            }
        )

        print(
            f"Epoch {epoch + 1}/{n_epochs} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
            print("  Best checkpoint updated")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    model.load_state_dict(
        torch.load(checkpoint_path, map_location=device, weights_only=True)
    )
    model.eval()
    print(f"Restored best checkpoint: epoch {best_epoch}, val loss {best_val_loss:.4f}")
    return losses, pd.DataFrame(history)


if RUN_TRAINING:
    val_loss_dl = DataLoader(
        validation_features,
        collate_fn=qa_loss_collate_fn,
        batch_size=32,
        shuffle=False,
    )

    losses, history = train(
        model,
        train_dl,
        val_loss_dl,
        optimizer,
        scheduler,
        n_epochs,
        device,
        patience=3,
        checkpoint_path=os.path.join(SAVE_PATH, "best_state_dict.pt"),
    )


In [9]:
if RUN_TRAINING:
    os.makedirs(SAVE_PATH, exist_ok=True)
    model.save_pretrained(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)
    history.to_csv(os.path.join(SAVE_PATH, "training_history.csv"), index=False)
    print(f"Saved best fine-tuned model and training history to {SAVE_PATH}")
else:
    print(f"Using previously saved model from {SAVE_PATH}")


Using previously saved model from ../checkpoints/integrated_qa_translated


In [10]:
validation_prediction_features = [
    {
        "input_ids": feature["input_ids"],
        "attention_mask": feature["attention_mask"],
        "example_index": feature["example_index"],
        "offset_mapping": [
            offset if seq_id == 1 else None
            for offset, seq_id in zip(
                feature["offset_mapping"], feature["sequence_ids"]
            )
        ],
    }
    for feature in validation_features
]

val_dl = DataLoader(
    validation_prediction_features,
    collate_fn=val_collate_fn,
    batch_size=32,
)


In [11]:
def predict(
    model,
    valid_dl,
    device,
):
    model.eval()

    start_logits_all = []
    end_logits_all = []

    with torch.no_grad():

        for batch in tqdm(valid_dl, desc="Evaluation"):

            batch = {key: value.to(device) for key, value in batch.items()}

            outputs = model(
                input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]
            )

            start_logits_all.extend(outputs.start_logits.detach().cpu().numpy())

            end_logits_all.extend(outputs.end_logits.detach().cpu().numpy())

    return (start_logits_all, end_logits_all)


def post_process_predictions(
    examples,
    dataset,
    logits,
    threshold=0.0,
    num_possible_answers=20,
    max_answer_length=30,
):

    all_start_logits, all_end_logits = logits

    features_per_example = defaultdict(list)

    for i, feature in enumerate(dataset):
        features_per_example[feature["example_index"]].append(i)

    predictions = []

    for example_idx, sample in examples.reset_index(drop=True).iterrows():

        feature_indices = features_per_example[example_idx]

        context = sample["context"]

        best_span_score = -np.inf
        best_span_text = ""

        best_null_score = np.inf

        for feature_idx in feature_indices:

            start_logits = all_start_logits[feature_idx]

            end_logits = all_end_logits[feature_idx]

            offset_mapping = dataset[feature_idx]["offset_mapping"]

            # CLS = no answer
            null_score = start_logits[0] + end_logits[0]

            best_null_score = min(best_null_score, null_score)

            start_indices = np.argsort(start_logits)[::-1][:num_possible_answers]

            end_indices = np.argsort(end_logits)[::-1][:num_possible_answers]

            for start_index in start_indices:
                for end_index in end_indices:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue

                    if (
                        start_index > end_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    score = start_logits[start_index] + end_logits[end_index]

                    if score > best_span_score:
                        start_char = offset_mapping[start_index][0]

                        end_char = offset_mapping[end_index][1]

                        best_span_score = score

                        best_span_text = context[start_char:end_char]

        no_answer_score = best_null_score - best_span_score

        prediction = "" if no_answer_score > threshold else best_span_text

        predictions.append(
            {
                "example_index": example_idx,
                "prediction": prediction,
                "no_answer_score": no_answer_score,
            }
        )

    return pd.DataFrame(predictions)


logits = predict(model, val_dl, device)

Evaluation: 100%|██████████| 38/38 [01:08<00:00,  1.80s/it]


In [12]:
predictions = post_process_predictions(
    df_val_translated,
    validation_prediction_features,
    logits,
    threshold=np.inf,
)
predictions.head()

,example_index,prediction,no_answer_score
0,0,Royal Academy of Åbo,-10.521364
1,1,"Finland, Sweden, Denmark",-9.701571
2,2,Grantham,-4.405603
3,3,Jeddah,6.795421
4,4,Abdrabbuh Mansur Hadi,-13.562586


In [13]:
val_examples = df_val_translated.reset_index(drop=True)

predictions["gold_answer"] = predictions["example_index"].map(
    lambda i: (
        str(val_examples.loc[i, "answer"]) if val_examples.loc[i, "answerable"] else ""
    )
)

predictions["gold_answerable"] = predictions["example_index"].map(
    lambda i: int(val_examples.loc[i, "answerable"])
)

predictions["lang"] = predictions["example_index"].map(
    lambda i: val_examples.loc[i, "lang"]
)


In [14]:
threshold_results = []

thresholds = np.linspace(
    predictions["no_answer_score"].min(),
    predictions["no_answer_score"].max(),
    100,
)

for threshold in thresholds:

    tuned_predictions = np.where(
        predictions["no_answer_score"] > threshold,
        "",
        predictions["prediction"],
    )

    em = np.mean(
        [
            exact_match_score(pred, gold)
            for pred, gold in zip(tuned_predictions, predictions["gold_answer"])
        ]
    )

    token_f1 = np.mean(
        [
            f1_score_squad(pred, gold)
            for pred, gold in zip(tuned_predictions, predictions["gold_answer"])
        ]
    )

    threshold_results.append(
        {
            "threshold": threshold,
            "EM": em,
            "Token F1": token_f1,
        }
    )

threshold_results = pd.DataFrame(threshold_results)


In [15]:
best_row = threshold_results.loc[threshold_results["Token F1"].idxmax()]

best_threshold = best_row["threshold"]

print(best_row)

threshold    0.717695
EM           0.509957
Token F1     0.610982
Name: 69, dtype: float64


In [16]:
predictions["prediction_tuned"] = np.where(
    predictions["no_answer_score"] > best_threshold,
    "",
    predictions["prediction"],
)

In [17]:
predictions["EM"] = [
    exact_match_score(pred, gold)
    for pred, gold in zip(predictions["prediction_tuned"], predictions["gold_answer"])
]

predictions["Token F1"] = [
    f1_score_squad(pred, gold)
    for pred, gold in zip(predictions["prediction_tuned"], predictions["gold_answer"])
]

predictions["pred_answerable"] = (predictions["prediction_tuned"].str.len() > 0).astype(
    int
)

xlmr_answerability_results = evaluate_answerability_predictions(
    y_true=predictions["gold_answerable"].astype(int),
    y_pred=predictions["pred_answerable"],
    languages=predictions["lang"],
    model_name="Integrated XLM-R QA",
)

In [18]:
print("SQuAD-v2 Exact Match:", predictions["EM"].mean())
print("SQuAD-v2 Token F1:", predictions["Token F1"].mean())

display(xlmr_answerability_results[["Language", "N", "Macro F1"]])

SQuAD-v2 Exact Match: 0.5099567099567099
SQuAD-v2 Token F1: 0.610981645518679


,Language,N,Macro F1
0,Overall,1155,0.770807
1,Arabic,415,0.929851
2,Korean,356,0.860556
3,Telugu,384,0.554352


In [19]:
lang_names = {"ar": "Arabic", "ko": "Korean", "te": "Telugu"}

by_language = (
    predictions.groupby("lang")
    .agg(
        N=("EM", "size"),
        EM=("EM", "mean"),
        Token_F1=("Token F1", "mean"),
    )
    .reset_index()
)
by_language["Language"] = by_language["lang"].map(lang_names)
by_language = by_language.drop(columns="lang")

overall = pd.DataFrame([{
    "Language": "Overall",
    "N": len(predictions),
    "EM": predictions["EM"].mean(),
    "Token_F1": predictions["Token F1"].mean(),
}])

qa_results = pd.concat([overall, by_language], ignore_index=True)
qa_results = qa_results.merge(
    xlmr_answerability_results[["Language", "Macro F1"]],
    on="Language",
    how="left",
)

print("Selected no-answer threshold:", best_threshold)
display(qa_results[["Language", "N", "EM", "Token_F1", "Macro F1"]].round(4))


Selected no-answer threshold: 0.7176952362060547


,Language,N,EM,Token_F1,Macro F1
0,Overall,1155,0.5100,0.6110,0.7708
1,Arabic,415,0.5373,0.6487,0.9299
2,Korean,356,0.5730,0.6713,0.8606
3,Telugu,384,0.4219,0.5143,0.5544


In [20]:
MODEL_TAG = "xlmr_nllb"
REVIEW_DIR = Path("../results/qa_analysis")
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

val_review = val_examples.reset_index(drop=True)
qa_review = predictions.copy()
qa_review = qa_review.merge(
    val_review[["question", "context"]].reset_index().rename(
        columns={"index": "example_index"}
    ),
    on="example_index",
    validate="one_to_one",
)
qa_review["pred_answerable"] = qa_review["pred_answerable"].astype(int)
qa_review["gold_answerable"] = qa_review["gold_answerable"].astype(int)

gold_yes = qa_review["gold_answerable"].eq(1)
pred_yes = qa_review["pred_answerable"].eq(1)
qa_review["error_type"] = np.select(
    [~gold_yes & pred_yes,
     gold_yes & ~pred_yes,
     gold_yes & pred_yes & qa_review["EM"].lt(1)],
    ["false_positive", "false_negative", "incorrect_span"],
    default="correct",
)

cols = [
    "example_index", "lang", "question", "context", "gold_answerable",
    "gold_answer", "prediction_tuned", "pred_answerable", "EM", "Token F1",
    "no_answer_score", "error_type",
]
all_path = REVIEW_DIR / f"{MODEL_TAG}_validation.csv"
qa_review[cols].sort_values("example_index").to_csv(
    all_path, index=False, encoding="utf-8-sig"
)

errors = qa_review[qa_review["error_type"] != "correct"].copy()
print("All validation errors by type and language:")
display(pd.crosstab(errors["lang"], errors["error_type"]))

selected_ids = []
for lang in ["ar", "ko", "te"]:
    pool = errors[errors["lang"] == lang]
    picked = []
    for error_type in ["false_positive", "false_negative", "incorrect_span"]:
        choices = pool[
            (pool["error_type"] == error_type)
            & (~pool["example_index"].isin(picked))
        ]
        if not choices.empty:
            picked.append(int(choices.sample(1, random_state=42)["example_index"].iloc[0]))
    choices = pool[~pool["example_index"].isin(picked)]
    n_remaining = max(0, 4 - len(picked))
    if n_remaining > 0 and not choices.empty:
        picked.extend(
            choices.sample(n=min(n_remaining, len(choices)), random_state=43)["example_index"].astype(int).tolist()
        )
    selected_ids.extend(picked)

selected_errors = qa_review[
    qa_review["example_index"].isin(selected_ids)
].sort_values(["lang", "error_type", "example_index"]).copy()
selected_errors["manual_category"] = ""
selected_errors["notes"] = ""
sample_path = REVIEW_DIR / f"{MODEL_TAG}_sampled_errors.csv"
selected_errors[cols + ["manual_category", "notes"]].to_csv(
    sample_path, index=False, encoding="utf-8-sig"
)
print(f"Selected {len(selected_errors)} errors; details saved to {sample_path}")
for _, row in selected_errors.iterrows():
    print("\n" + "=" * 90)
    print(f"INDEX {row['example_index']} | {row['lang']} | {row['error_type']} | "
          f"EM {row['EM']:.2f} | Token F1 {row['Token F1']:.2f}")
    print("QUESTION:", row["question"])
    print("GOLD:", repr(row["gold_answer"]))
    print("PREDICTED:", repr(row["prediction_tuned"]))
    print("CONTEXT:", row["context"])
print("\nFull validation export:", all_path)


All validation errors by type and language:


error_type,false_negative,false_positive,incorrect_span
lang,,,
ar,13,1,178
ko,10,2,140
te,4,80,138


Selected 12 errors; details saved to ../results/qa_analysis/xlmr_nllb_sampled_errors.csv

INDEX 363 | ar | false_negative | EM 0.00 | Token F1 0.00
QUESTION: Did the Argentine player Messi have Spanish citizenship before joining the Argentine national team?
GOLD: 'Spanish descent'
PREDICTED: ''
CONTEXT: Lionel Andrés Messi was born on 24 June 1987 in Rosario, the third of four children of Jorge Messi, a steel factory manager, and his wife Celia Cuccittini, who worked in a magnet manufacturing workshop. On his father's side, he is of Italian and Spanish descent, the great-grandson of immigrants from the northcentral Adriatic Marche region of Italy and Catalonia, and on his mother's side, he has primarily Italian ancestry. Growing up in a tight-knit, football-loving family, "Leo" developed a passion for the sport from an early age, playing constantly with his older brothers, Rodrigo and Matías, and his cousins, Maximiliano and Emanuel Biancucchi, both of whom became professional football

In [22]:
with open("../student_starter/challenge.json", encoding="utf-8") as f:
    challenge_data = json.load(f)

challenge_df = pd.DataFrame(challenge_data)

display(challenge_df[[
    "id", "pair_id", "lang", "question",
    "answerable", "answer"
]])

,id,pair_id,lang,question,answerable,answer
0,rapa-nui-answerable,rapa-nui,ar,ما الاسم الأصلي لجزيرة الفصح؟,True,Rapa Nui
1,rapa-nui-unanswerable,rapa-nui,ar,ما الاسم الإسباني لجزيرة الفصح؟,False,
2,chinchorro-answerable,chinchorro,ko,가장 오래된 인공 미라로 유명한 문화는 무엇인가요?,True,Chinchorro
3,chinchorro-unanswerable,chinchorro,ko,가장 오래된 자연 미라로 유명한 문화는 무엇인가요?,False,
4,alma-answerable,alma,te,ALMA యాంటెన్నాల మధ్య గరిష్ఠ దూరం ఎంత?,True,16 kilometres
5,alma-unanswerable,alma,te,ALMA యాంటెన్నాల మధ్య కనిష్ఠ దూరం ఎంత?,False,
6,chiloe-answerable,chiloe,ar,ما اسم الحصن التاريخي الذي يمكن زيارته في أنكو...,True,Fort San Antonio
7,chiloe-unanswerable,chiloe,ar,ما اسم الحصن التاريخي الذي يمكن زيارته في كاست...,False,
8,uyuni-answerable,uyuni,ko,세계에서 가장 큰 소금 평원의 이름은 무엇인가요?,True,Salar de Uyuni
9,uyuni-unanswerable,uyuni,ko,세계에서 두 번째로 큰 소금 평원의 이름은 무엇인가요?,False,


In [23]:
translator = NLLBTranslator()

challenge_translated = challenge_df.copy()

for lang in ["ar", "ko", "te"]:
    indices = challenge_df.index[challenge_df["lang"] == lang]

    questions = challenge_df.loc[indices, "question"].tolist()

    translations = translator.translate_batch(
        questions,
        source_lang=lang,
        batch_size=8,
    )

    challenge_translated.loc[indices, "question"] = translations

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

Translating ar:   0%|          | 0/1 [00:00<?, ?it/s]

Translating ko:   0%|          | 0/1 [00:00<?, ?it/s]

Translating te:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
def evaluate_challenge(model, df, threshold):
    df = df.reset_index(drop=True).copy()

    features = build_qa_features(
        df,
        tokenizer,
        max_length=MAX_LENGTH,
        doc_stride=DOC_STRIDE,
    )

    prediction_features = [
        {
            "input_ids": f["input_ids"],
            "attention_mask": f["attention_mask"],
            "example_index": f["example_index"],
            "offset_mapping": [
                offset if sid == 1 else None
                for offset, sid in zip(
                    f["offset_mapping"], f["sequence_ids"]
                )
            ],
        }
        for f in features
    ]

    challenge_dl = DataLoader(
        prediction_features,
        collate_fn=val_collate_fn,
        batch_size=32,
        shuffle=False,
    )

    logits = predict(model, challenge_dl, device)

    raw_predictions = post_process_predictions(
        df,
        prediction_features,
        logits,
        threshold=threshold,
    )

    results = df[[
        "id", "pair_id", "lang",
        "question", "answerable", "answer"
    ]].copy()

    results["prediction"] = raw_predictions["prediction"].to_numpy()

    results["EM"] = [
        exact_match_score(pred, gold)
        for pred, gold in zip(
            results["prediction"], results["answer"]
        )
    ]

    results["Token F1"] = [
        f1_score_squad(pred, gold)
        for pred, gold in zip(
            results["prediction"], results["answer"]
        )
    ]

    results["pred_answerable"] = (
        results["prediction"].str.len() > 0
    )

    results["answerability_correct"] = (
        results["pred_answerable"] == results["answerable"]
    )

    return results

In [25]:
challenge_results_translated = evaluate_challenge(
    model,
    challenge_translated,
    threshold=best_threshold,
)

display(challenge_results_translated[[
    "id",
    "answer",
    "prediction",
    "EM",
    "Token F1",
    "answerability_correct",
]])

Evaluation: 100%|██████████| 1/1 [00:00<00:00,  4.80it/s]


,id,answer,prediction,EM,Token F1,answerability_correct
0,rapa-nui-answerable,Rapa Nui,Rapa Nui,True,1.0,True
1,rapa-nui-unanswerable,,Rapa Nui,False,0.0,False
2,chinchorro-answerable,Chinchorro,Chinchorro,True,1.0,True
3,chinchorro-unanswerable,,Chinchorro,False,0.0,False
4,alma-answerable,16 kilometres,16 kilometres,True,1.0,True
5,alma-unanswerable,,16 kilometres,False,0.0,False
6,chiloe-answerable,Fort San Antonio,Fort San Antonio,True,1.0,True
7,chiloe-unanswerable,,Fort San Antonio,False,0.0,False
8,uyuni-answerable,Salar de Uyuni,Salar de Uyuni,True,1.0,True
9,uyuni-unanswerable,,Salar de Uyuni,False,0.0,False


In [26]:
def challenge_summary(results):
    pair_results = (
        results.groupby("pair_id")
        .agg(
            lang=("lang", "first"),
            answerability_both_correct=(
                "answerability_correct", "all"
            ),
            exact_both_correct=("EM", "all"),
        )
        .reset_index()
    )

    print("SQuAD-v2 EM:", results["EM"].mean())
    print("SQuAD-v2 Token F1:", results["Token F1"].mean())

    print(
        "Answerability Macro-F1:",
        f1_score(
            results["answerable"].astype(int),
            results["pred_answerable"].astype(int),
            labels=[0, 1],
            average="macro",
            zero_division=0,
        ),
    )

    print(
        "Pairs with correct answerability:",
        pair_results["answerability_both_correct"].sum(),
        "/", len(pair_results),
    )

    print(
        "Pairs with exact QA:",
        pair_results["exact_both_correct"].sum(),
        "/", len(pair_results),
    )

    display(pair_results)

    return pair_results


challenge_pairs_translated = challenge_summary(
    challenge_results_translated
)

SQuAD-v2 EM: 0.5
SQuAD-v2 Token F1: 0.5
Answerability Macro-F1: 0.3333333333333333
Pairs with correct answerability: 0 / 10
Pairs with exact QA: 0 / 10


,pair_id,lang,answerability_both_correct,exact_both_correct
0,alma,te,False,False
1,angkor-wat,ko,False,False
2,chiloe,ar,False,False
3,chinchorro,ko,False,False
4,eiffel-tower,ar,False,False
5,great-barrier-reef,te,False,False
6,mount-fuji,ar,False,False
7,rapa-nui,ar,False,False
8,taj-mahal,ko,False,False
9,uyuni,ko,False,False
